In [ ]:
# PART 1: SETUP AND INSTALLATION

# Clean up and install required packages with compatible versions
!pip uninstall -y sentence-transformers
!pip install -q transformers datasets accelerate peft trl bitsandbytes wandb sentencepiece protobuf

In [ ]:
pip install --upgrade transformers trl bitsandbytes accelerate peft datasets

In [8]:
import torch
torch.cuda.empty_cache()
import gc

# Assuming `large_variable` is consuming a lot of memory
gc.collect() # Collects the garbage, freeing up memory


0

In [14]:
import torch
import os
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
import transformers
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig
)
from datasets import load_dataset, Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import numpy as np
import json
from typing import List, Dict
import wandb
import contextlib


print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Configuration
QUICK_TEST = True  # Set to False for full training

# ============================================================================
# PART 2: DATA PREPARATION - MEDICAL REASONING DATASET
# ============================================================================

def prepare_medical_dataset():
    """
    Load and prepare MedQA dataset for medical reasoning fine-tuning
    MedQA contains USMLE-style medical questions with detailed explanations
    """
    print("Loading MedQA dataset...")

    # Load MedQA from HuggingFace
    dataset = load_dataset("GBaker/MedQA-USMLE-4-options", split="train")

    # Adjust dataset size based on QUICK_TEST mode
    if QUICK_TEST:
        num_samples = 500  # Quick test
        print("⚡ QUICK TEST MODE: Using 500 samples")
    else:
        num_samples = 2000  # Full training
        print("🚀 FULL TRAINING MODE: Using 2000 samples")

    dataset = dataset.select(range(min(num_samples, len(dataset))))

    def format_prompt(example):
        """Format medical question as instruction-following prompt"""
        question = example['question']
        options = example['options']

        # Create formatted options string
        options_text = "\n".join([f"{k}: {v}" for k, v in options.items()])

        prompt = f"""You are a medical expert. Answer the following medical question with detailed reasoning.

Question: {question}

Options:
{options_text}

Provide your answer in the following format:
1. Reasoning: Step-by-step medical reasoning
2. Answer: The correct option (A/B/C/D)

Response:"""

        # Create target response
        answer_key = example['answer_idx']
        answer_text = options[answer_key]

        # Use existing reasoning if available, otherwise create template
        reasoning = example.get('reasoning', 'Based on medical knowledge and clinical guidelines, analyzing the symptoms and presentation leads to the diagnosis.')

        response = f"""1. Reasoning: {reasoning}

2. Answer: {answer_key} - {answer_text}"""

        return {
            'prompt': prompt,
            'response': response,
            'completion': response,
            # This 'text' field is crucial for SFTTrainer
            'text': prompt + response
        }

    formatted_dataset = dataset.map(format_prompt, remove_columns=dataset.column_names)

    # Split into train/validation
    split_dataset = formatted_dataset.train_test_split(test_size=0.15, seed=42)

    print(f"Training samples: {len(split_dataset['train'])}")
    print(f"Validation samples: {len(split_dataset['test'])}")

    return split_dataset['train'], split_dataset['test']

# ============================================================================
# PART 3: MODEL SETUP WITH QUANTIZATION
# ============================================================================
# Avoid CUDA fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

def setup_model_and_tokenizer(model_name="microsoft/Phi-3-mini-4k-instruct"):
    print(f"Loading model: {model_name}")
    
    # MODIFICATION 1: Use a more stable quantization config for T4 GPUs
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        # Use float16 as the compute dtype, which is better supported on T4s
        bnb_4bit_compute_dtype=torch.float16, 
        # Disable double quantization as a stability measure
        bnb_4bit_use_double_quant=False,
    )

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
        padding_side='right'
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    n_gpus = torch.cuda.device_count()
    print("Visible GPUs:", n_gpus)
    for i in range(n_gpus):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

    max_mem = {}
    if n_gpus > 0:
        for i in range(n_gpus):
            mem_gb = int(torch.cuda.get_device_properties(i).total_memory / 1024**3)
            headroom = 2
            allowed = max(1, mem_gb - headroom)
            max_mem[i] = f"{allowed}GB"
        max_mem["cpu"] = "60GB"

    offload_folder = "/tmp/model_offload"
    os.makedirs(offload_folder, exist_ok=True)

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        max_memory=max_mem,
        offload_folder=offload_folder,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
        # MODIFICATION 2: Match the torch_dtype to the compute_dtype
        torch_dtype=torch.float16, 
    )
    
    model = prepare_model_for_kbit_training(model)

    print("Model loaded successfully.")
    return model, tokenizer


# ============================================================================
# PART 4: LORA CONFIGURATION FOR EFFICIENT FINE-TUNING
# ============================================================================

def get_lora_config():
    """
    Configure LoRA (Low-Rank Adaptation) for parameter-efficient fine-tuning
    """
    lora_config = LoraConfig(
        r=16,  # LoRA rank
        lora_alpha=32,  # LoRA scaling
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj"
        ],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    return lora_config

# ============================================================================
# PART 5: SUPERVISED FINE-TUNING (SFT)
# ============================================================================
# ============================================================================
def count_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total


def run_supervised_finetuning(model, tokenizer, train_dataset, eval_dataset):
    """
    Phase 1: Supervised Fine-Tuning on medical reasoning data.

    - Wraps model with LoRA (using get_lora_config() + get_peft_model)
    - Builds an SFTConfig (preferred) or falls back to TrainingArguments
    - Instantiates SFTTrainer with tokenizer and peft_config
    - Trains and returns (model, tokenizer, trainer)
    """
    print("\n" + "="*60)
    print("PHASE 1: SUPERVISED FINE-TUNING")
    print("="*60)

    # === Prepare LoRA / PEFT ===
    # get_lora_config() is your helper that returns a LoraConfig()
    lora_config = get_lora_config()

    # Wrap model with PEFT (LoRA)
    # Use get_peft_model (from peft) to apply the LoraConfig to the model
    model = get_peft_model(model, lora_config)

    # === Print parameter counts ===
    trainable, total = count_trainable_params(model)
    total = total if total > 0 else 1  # avoid division by zero
    print(f"Trainable parameters: {trainable / 1e6:.2f}M")
    print(f"All parameters: {total / 1e9:.2f}B")
    print(f"Trainable %: {100 * trainable / total:.4f}%")

    # === Device / dtype settings ===
    epochs = 1 if QUICK_TEST else 2

    bf16_supported = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    fp16_supported = torch.cuda.is_available() and not bf16_supported
    if not bf16_supported:
        print("⚠️  BF16 not supported on this GPU, using FP16 if CUDA available.")

    model.config.use_cache = False  # required for some trainer flows

    # === Build SFTConfig (preferred) or TrainingArguments fallback ===
    sft_args = None
    try:
        from trl import SFTConfig

        sft_args = SFTConfig(
            output_dir="./medical-phi3-sft",
            num_train_epochs=epochs,
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=8,
            learning_rate=2e-4,
            warmup_steps=50,
            logging_steps=20,
            logging_strategy="steps",
            do_eval=True,
            eval_steps=200,
            save_strategy="steps",
            save_steps=500,
            save_total_limit=1,
            fp16=fp16_supported,
            bf16=bf16_supported,
            optim="paged_adamw_8bit",
            max_grad_norm=0.3,
            lr_scheduler_type="cosine",
            report_to="none",
            load_best_model_at_end=False,
            gradient_checkpointing=True,
            dataset_text_field="text",
            max_seq_length=1024,
            packing=False,
        )
        print("Using trl.SFTConfig for trainer arguments.")
    except Exception as e:
        print("trl.SFTConfig not available or failed to import — falling back to transformers.TrainingArguments.")
        from transformers import TrainingArguments

        sft_args = TrainingArguments(
            output_dir="./medical-phi3-sft",
            num_train_epochs=epochs,
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=8,
            learning_rate=2e-4,
            warmup_steps=50,
            logging_steps=20,
            logging_strategy="steps",
            do_eval=True,
            eval_steps=200,
            save_strategy="steps",
            save_steps=500,
            save_total_limit=1,
            fp16=fp16_supported,
            bf16=bf16_supported,
            optim="paged_adamw_8bit",
            max_grad_norm=0.3,
            lr_scheduler_type="cosine",
            report_to="none",
            load_best_model_at_end=False,
        )

    # === Initialize SFTTrainer with tokenizer and peft_config ===
    # Use a dictionary to handle argument name changes in trl
    trainer_kwargs = {
        "model": model,
        "args": sft_args,
        "train_dataset": train_dataset,
        "eval_dataset": eval_dataset,
        "peft_config": lora_config,
    }
    
    # Conditionally set `processing_class` for newer trl versions
    # or `tokenizer` for older ones if a fallback is needed.
    # The most robust approach for modern trl is to use `processing_class`.
    from trl.trainer.sft_trainer import SFTTrainer
    trainer_kwargs["processing_class"] = tokenizer

    trainer = SFTTrainer(**trainer_kwargs)

    # === Train ===
    print("\nStarting supervised fine-tuning...")
    trainer.train()

    # === Save ===
    try:
        trainer.save_model("./medical-phi3-sft-final")
    except Exception:
        print("trainer.save_model failed; attempting model.save_pretrained + tokenizer.save_pretrained")
        model.save_pretrained("./medical-phi3-sft-final")
        tokenizer.save_pretrained("./medical-phi3-sft-final")

    print("\nSFT complete.")
    return model, tokenizer, trainer



# ============================================================================
# PART 6: GRPO - CUSTOM IMPLEMENTATION
# ============================================================================

def prepare_preference_dataset(eval_dataset, model, tokenizer, num_samples=200):
    """
    Generate preference pairs for GRPO training
    """
    print("\n" + "="*60)
    print("PREPARING PREFERENCE DATASET FOR GRPO")
    print("="*60)

    if QUICK_TEST:
        num_samples = 50
        print("⚡ QUICK TEST MODE: Creating 50 preference pairs")
    else:
        num_samples = 200
        print("🚀 FULL MODE: Creating 200 preference pairs")

    preference_data = []
    num_to_process = min(num_samples, len(eval_dataset))

    model.eval()

    for i in range(num_to_process):
        if i % 20 == 0:
            print(f"Processing example {i}/{num_to_process}...")

        example = eval_dataset[i]
        prompt = example['prompt']

        # Tokenize
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)

        responses = []
        with torch.no_grad():
            for _ in range(3):
                outputs = model.generate(
                    input_ids=inputs['input_ids'],
                    attention_mask=inputs['attention_mask'],
                    max_new_tokens=200,
                    do_sample=True,
                    temperature=0.8,
                    top_p=0.9,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id
                )
                new_tokens = outputs[0, inputs['input_ids'].shape[-1]:]
                response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
                responses.append(response)

        preference_data.append({
            'prompt': prompt,
            'chosen': example['response'],
            'rejected': responses[0]
        })

    print(f"Created {len(preference_data)} preference pairs")
    return Dataset.from_list(preference_data)


class SimpleGRPOTrainer:
    """
    Simplified GRPO implementation for quantized models
    """
    def __init__(self, model, tokenizer, train_dataset, learning_rate=5e-6,
                 batch_size=1, num_epochs=1, beta=0.1):
        self.model = model
        self.tokenizer = tokenizer
        self.train_dataset = train_dataset
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.num_epochs = num_epochs
        self.beta = beta

        # Setup optimizer
        self.optimizer = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=learning_rate
        )


    def _sequence_logprob(self, logits, labels, ignore_index=None):
        """
        logits: [B, T, V]
        labels: [B, T] (token ids for sequence)
        returns: tensor shape [B] sequence log-prob (sum over tokens)
        """
        # compute log-probs per token
        logp = torch.nn.functional.log_softmax(logits, dim=-1)  # [B,T,V]
        # gather log-prob of the labels
        labels_expanded = labels.unsqueeze(-1)  # [B, T, 1]
        token_logps = torch.gather(logp, dim=-1, index=labels_expanded).squeeze(-1)  # [B, T]
        if ignore_index is not None:
            mask = (labels != ignore_index).float()
            # sum only over masked tokens
            seq_logp = (token_logps * mask).sum(dim=-1)
        else:
            seq_logp = token_logps.sum(dim=-1)
        return seq_logp


    def compute_loss(self, prompt, chosen, rejected):
        # Build inputs and labels (we compute logprob of entire prompt+response sequence)
        chosen_text = prompt + chosen
        rejected_text = prompt + rejected

        chosen_input = self.tokenizer(chosen_text, return_tensors="pt", truncation=True, max_length=1024)
        rejected_input = self.tokenizer(rejected_text, return_tensors="pt", truncation=True, max_length=1024)

        chosen_outputs = self.model(
            input_ids=chosen_input['input_ids'],
            attention_mask=chosen_input['attention_mask'],
        )
        rejected_outputs = self.model(
            input_ids=rejected_input['input_ids'],
            attention_mask=rejected_input['attention_mask'],
        )

        chosen_seq_logp = self._sequence_logprob(chosen_outputs.logits, chosen_input['input_ids'])
        rejected_seq_logp = self._sequence_logprob(rejected_outputs.logits, rejected_input['input_ids'])

        diff = self.beta * (chosen_seq_logp - rejected_seq_logp)
        loss = -torch.logsigmoid(diff).mean()
        return loss

    def train(self):
        """Train with preference optimization"""
        self.model.train()

        print(f"\nStarting GRPO training for {self.num_epochs} epoch(s)...")
        print(f"Training samples: {len(self.train_dataset)}")

        total_steps = len(self.train_dataset) * self.num_epochs
        step = 0

        for epoch in range(self.num_epochs):
            print(f"\nEpoch {epoch + 1}/{self.num_epochs}")
            epoch_loss = 0

            for i, example in enumerate(self.train_dataset):
                prompt = example['prompt']
                chosen = example['chosen']
                rejected = example['rejected']

                loss = self.compute_loss(prompt, chosen, rejected)

                self.optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.optimizer.step()

                epoch_loss += loss.item()
                step += 1

                if step % 10 == 0:
                    avg_loss = epoch_loss / (i + 1)
                    print(f"Step {step}/{total_steps} | Loss: {avg_loss:.4f}")

            avg_epoch_loss = epoch_loss / len(self.train_dataset)
            print(f"Epoch {epoch + 1} completed | Average Loss: {avg_epoch_loss:.4f}")

        print("\nGRPO training completed!")
        return self.model


def run_grpo_training(model, tokenizer, preference_dataset):
    """
    Phase 2: GRPO Training
    """
    print("\n" + "="*60)
    print("PHASE 2: GRPO TRAINING")
    print("="*60)

    grpo_trainer = SimpleGRPOTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=preference_dataset,
        learning_rate=5e-6,
        batch_size=1,
        num_epochs=1,
        beta=0.1
    )

    model = grpo_trainer.train()

    model.save_pretrained("./medical-phi3-grpo-final")
    tokenizer.save_pretrained("./medical-phi3-grpo-final")

    print("\nGRPO model saved to ./medical-phi3-grpo-final")
    return model

# ============================================================================
# PART 7: EVALUATION
# ============================================================================

def evaluate_model(model, tokenizer, test_prompts: List[str], model_name: str):
    """
    Evaluate model on test prompts
    """
    print(f"\n{'='*60}")
    print(f"EVALUATING: {model_name}")
    print(f"{'='*60}\n")

    results = []
    model.eval()

    for i, prompt in enumerate(test_prompts):
        print(f"\n--- Test Case {i+1} ---")
        print(f"Prompt: {prompt[:100]}...")

        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)

        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                max_new_tokens=300,
                do_sample=True,
                temperature=0.7,
                top_p=0.95,
                pad_token_id=tokenizer.pad_token_id
            )
        new_tokens = outputs[0, inputs['input_ids'].shape[-1]:]
        response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

        print(f"\nResponse:\n{response}\n")

        results.append({
            'prompt': prompt,
            'response': response
        })

    return results

def run_complete_pipeline():
    """
    Execute the complete fine-tuning pipeline
    """
    print("="*60)
    print("MEDICAL REASONING LLM FINE-TUNING WITH GRPO")
    print("="*60)

    # Step 1: Prepare dataset
    train_dataset, eval_dataset = prepare_medical_dataset()

    # Step 2: Setup model
    model, tokenizer = setup_model_and_tokenizer()

    # Step 3: Supervised Fine-Tuning
    model, tokenizer, trainer = run_supervised_finetuning(model, tokenizer, train_dataset, eval_dataset)

    # Step 4: Prepare preference dataset
    preference_dataset = prepare_preference_dataset(eval_dataset, model, tokenizer)

    # Step 5: GRPO Training
    model = run_grpo_training(model, tokenizer, preference_dataset)

    # Step 6: Evaluation
    test_prompts = [
        """You are a medical expert. Answer the following medical question with detailed reasoning.

Question: A 45-year-old man presents with chest pain radiating to his left arm and jaw. What is the most likely diagnosis?

Options:
A: Gastroesophageal reflux disease
B: Acute myocardial infarction
C: Costochondritis
D: Panic attack

Provide your answer in the following format:
1. Reasoning: Step-by-step medical reasoning
2. Answer: The correct option (A/B/C/D)

Response:""",

        """You are a medical expert. Answer the following medical question with detailed reasoning.

Question: A 28-year-old pregnant woman in her third trimester presents with severe headache and visual disturbances. Her blood pressure is 160/110 mmHg. What is the most appropriate immediate management?

Options:
A: Oral antihypertensives and outpatient follow-up
B: IV magnesium sulfate and blood pressure control
C: Immediate cesarean section
D: Diuretics and bed rest

Provide your answer in the following format:
1. Reasoning: Step-by-step medical reasoning
2. Answer: The correct option (A/B/C/D)

Response:"""
    ]

    results = evaluate_model(model, tokenizer, test_prompts, "Medical-Phi3-GRPO")

    print("\n" + "="*60)
    print("PIPELINE COMPLETED SUCCESSFULLY!")
    print("="*60)
    print(f"\nModels saved:")
    print("- Supervised FT: ./medical-phi3-sft-final")
    print("- GRPO: ./medical-phi3-grpo-final")

    return model, tokenizer, results

PyTorch version: 2.6.0+cu124
Transformers version: 4.57.0
CUDA available: True
GPU: Tesla T4


In [11]:
# ============================================================================
# EXECUTE PIPELINE
# ============================================================================

if __name__ == "__main__":
    model, tokenizer, results = run_complete_pipeline()

    with open('evaluation_results.json', 'w') as f:
        json.dump(results, f, indent=2)

    print("\nResults saved to evaluation_results.json")

MEDICAL REASONING LLM FINE-TUNING WITH GRPO
Loading MedQA dataset...
⚡ QUICK TEST MODE: Using 500 samples
Training samples: 425
Validation samples: 75
Loading model: microsoft/Phi-3-mini-4k-instruct


`torch_dtype` is deprecated! Use `dtype` instead!


Visible GPUs: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded successfully.

PHASE 1: SUPERVISED FINE-TUNING
Trainable parameters: 8.91M
All parameters: 2.02B
Trainable %: 0.4417%
trl.SFTConfig not available or failed to import — falling back to transformers.TrainingArguments.


...


Starting supervised fine-tuning...


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
20,1.389800
40,0.240700



SFT complete.


ValueError: too many values to unpack (expected 2)

In [12]:
import os

print("SFT dir exists?", os.path.isdir("./medical-phi3-sft-final"))
print("Listing files:")
if os.path.isdir("./medical-phi3-sft-final"):
    print(os.listdir("./medical-phi3-sft-final"))


SFT dir exists? True
Listing files:
['training_args.bin', 'tokenizer.json', 'added_tokens.json', 'adapter_model.safetensors', 'special_tokens_map.json', 'adapter_config.json', 'tokenizer.model', 'tokenizer_config.json', 'chat_template.jinja', 'README.md']


In [21]:
# Paste & run this cell (single runnable block)

import os, torch, json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

SFT_DIR = "./medical-phi3-sft-final"    # adapter + tokenizer saved here
BASE_MODEL = "microsoft/Phi-3-mini-4k-instruct"
offload_folder = "/tmp/model_offload"
os.makedirs(offload_folder, exist_ok=True)

print("SFT_DIR exists:", os.path.isdir(SFT_DIR))
print("Contents (top-level):", os.listdir(SFT_DIR))

# 1) Tokenizer: prefer saved tokenizer from SFT_DIR so tokenization matches training
try:
    tokenizer = AutoTokenizer.from_pretrained(SFT_DIR, trust_remote_code=True, padding_side="right")
    print("Loaded tokenizer from SFT_DIR.")
except Exception as e:
    print("Could not load tokenizer from SFT_DIR, falling back to base tokenizer:", e)
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True, padding_side="right")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2) BitsAndBytes config (match training)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

# 3) Load base quantized model (device_map="auto" will distribute/load on GPUs)
print("Loading base quantized model (this may take ~30s)...")
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    offload_folder=offload_folder,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
print("Base model loaded.")

# 4) Attach saved LoRA adapter (PeftModel)
adapter_path = os.path.join(SFT_DIR, "adapter_model.safetensors")
if os.path.exists(adapter_path):
    print("Attaching LoRA adapter from:", SFT_DIR)
    model = PeftModel.from_pretrained(base, SFT_DIR, device_map="auto")
else:
    # If adapter missing in final dir, try checkpoint folder
    ckpt_dir = "./medical-phi3-sft/checkpoint-54"
    if os.path.exists(os.path.join(ckpt_dir, "adapter_model.safetensors")):
        print("Adapter not in final dir; attaching from checkpoint:", ckpt_dir)
        model = PeftModel.from_pretrained(base, ckpt_dir, device_map="auto")
    else:
        print("No adapter found; using base model (no SFT).")
        model = base

model.eval()
print("Model ready. CUDA available:", torch.cuda.is_available())

# 5) Recreate dataset objects if missing (calls your existing func)
if 'eval_dataset' not in globals() or 'train_dataset' not in globals():
    print("(Re)creating train & eval datasets via prepare_medical_dataset() ...")
    train_dataset, eval_dataset = prepare_medical_dataset()
else:
    train_dataset = globals()['train_dataset']
    eval_dataset = globals()['eval_dataset']
    print("Using in-memory datasets.")

# # 6) Create preference dataset (quick mode will keep this small)
# print("\nPreparing preference dataset (this calls model.generate() internally)...")
# preference_dataset = prepare_preference_dataset(eval_dataset, model, tokenizer)
# print("Preference dataset size:", len(preference_dataset))

# # 7) Run GRPO training (this updates adapter in-memory and saves to ./medical-phi3-grpo-final)
# print("\nStarting GRPO training (SimpleGRPOTrainer)...")
# model = run_grpo_training(model, tokenizer, preference_dataset)
# print("GRPO finished. Model saved to ./medical-phi3-grpo-final")

# 8) Evaluation prompts (same two you used)
test_prompts = [
    """You are a medical expert. Answer the following medical question with detailed reasoning.

Question: A 45-year-old man presents with chest pain radiating to his left arm and jaw. What is the most likely diagnosis?

Options:
A: Gastroesophageal reflux disease
B: Acute myocardial infarction
C: Costochondritis
D: Panic attack

Provide your answer in the following format:
1. Reasoning: Step-by-step medical reasoning
2. Answer: The correct option (A/B/C/D)

Response:""",
    """You are a medical expert. Answer the following medical question with detailed reasoning.

Question: A 28-year-old pregnant woman in her third trimester presents with severe headache and visual disturbances. Her blood pressure is 160/110 mmHg. What is the most appropriate immediate management?

Options:
A: Oral antihypertensives and outpatient follow-up
B: IV magnesium sulfate and blood pressure control
C: Immediate cesarean section
D: Diuretics and bed rest

Provide your answer in the following format:
1. Reasoning: Step-by-step medical reasoning
2. Answer: The correct option (A/B/C/D)

Response:"""
]

# 9) Run evaluation
print("\nRunning evaluation on test prompts...")
results = evaluate_model(model, tokenizer, test_prompts, "Medical-Phi3-GRPO")

# 10) Save results
with open('evaluation_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("Evaluation saved -> evaluation_results.json")

# Print abbreviated responses
for i, r in enumerate(results):
    print(f"\n--- Test {i+1} response (first 800 chars) ---\n{r['response'][:800]}\n")


SFT_DIR exists: True
Contents (top-level): ['training_args.bin', 'tokenizer.json', 'added_tokens.json', 'adapter_model.safetensors', 'special_tokens_map.json', 'adapter_config.json', 'tokenizer.model', 'tokenizer_config.json', 'chat_template.jinja', 'README.md']
Loaded tokenizer from SFT_DIR.
Loading base quantized model (this may take ~30s)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Base model loaded.
Attaching LoRA adapter from: ./medical-phi3-sft-final
Model ready. CUDA available: True
Using in-memory datasets.

Running evaluation on test prompts...

EVALUATING: Medical-Phi3-GRPO


--- Test Case 1 ---
Prompt: You are a medical expert. Answer the following medical question with detailed reasoning.

Question: ...


AttributeError: 'DynamicCache' object has no attribute 'seen_tokens'

In [20]:
!pip install --upgrade "transformers>=4.41.0"

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [22]:
# Robust generate helper + fixed evaluate_model
import torch

def generate_with_fallback(model, tokenizer, input_ids, attention_mask=None, **gen_kwargs):
    """
    Try model.generate normally; if the Phi-3 DynamicCache AttributeError occurs,
    retry with use_cache=False. Also attempts to move inputs to model device if safe.
    """
    # Determine model device (best-effort)
    try:
        first_param = next(model.parameters())
        model_device = first_param.device
    except StopIteration:
        model_device = None

    # Save original input length (before any .to())
    input_len = input_ids.shape[-1]

    # Best-effort move inputs to model_device to avoid CPU-vs-GPU warnings
    if model_device is not None:
        try:
            # Only move if devices differ
            if input_ids.device != model_device:
                input_ids = input_ids.to(model_device)
                if attention_mask is not None:
                    attention_mask = attention_mask.to(model_device)
        except Exception:
            # If moving fails (sharded / odd layout), continue with CPU tensors
            pass

    # Try normal generate
    try:
        outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, **gen_kwargs)
        return outputs, input_len
    except AttributeError as e:
        msg = str(e)
        if "seen_tokens" in msg or "DynamicCache" in msg:
            # fallback: retry with use_cache=False
            print("Caught DynamicCache AttributeError during generate(); retrying with use_cache=False.")
            outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, use_cache=False, **gen_kwargs)
            return outputs, input_len
        # re-raise if it's some other AttributeError
        raise

def evaluate_model(model, tokenizer, test_prompts: list, model_name: str):
    """
    Robust evaluation that handles Phi-3 DynamicCache error and device mismatches.
    Returns list of dicts with 'prompt' and 'response'.
    """
    print(f"\n{'='*60}")
    print(f"EVALUATING: {model_name}")
    print(f"{'='*60}\n")

    results = []
    model.eval()

    for i, prompt in enumerate(test_prompts):
        print(f"\n--- Test Case {i+1} ---")
        print(f"Prompt: {prompt[:200]}...")

        try:
            # Tokenize (keep tensors on CPU initially)
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512, padding=False)
            input_ids = inputs["input_ids"]
            attention_mask = inputs.get("attention_mask", None)

            # Generation kwargs
            gen_kwargs = dict(
    max_new_tokens=120,
    do_sample=False,             # deterministic greedy decode for evaluation
    temperature=0.0,
    top_p=1.0,
    repetition_penalty=1.2,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
    use_cache=False,             # safe if you hit DynamicCache problems
            )

            # Use helper to generate with fallback
            outputs, input_len = generate_with_fallback(model, tokenizer, input_ids, attention_mask=attention_mask, **gen_kwargs)

            # Ensure output is on CPU for decoding
            out_ids = outputs[0]
            if out_ids.device != torch.device("cpu"):
                out_ids = out_ids.cpu()

            # slice out generated tokens (avoid negative slicing issues)
            if out_ids.shape[-1] > input_len:
                new_tokens = out_ids[input_len:]
            else:
                new_tokens = out_ids.new_empty((0,))

            response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

            print(f"\nResponse:\n{response}\n")

            results.append({
                "prompt": prompt,
                "response": response
            })

        except Exception as e:
            # Catch-all to avoid crashing the entire evaluation run
            import traceback, sys
            tb = traceback.format_exc()
            print("Generation failed for this prompt. Exception:\n", tb)
            results.append({
                "prompt": prompt,
                "response": f"<Generation failed: {repr(e)}>"
            })

    return results

# ---------------------------
# Quick single-prompt test (run this cell and inspect output)
# ---------------------------
# Use one prompt to validate generation before running full evaluation
test_prompt = """You are a medical expert. Answer the following medical question with detailed reasoning.

Question: A 45-year-old man presents with chest pain radiating to his left arm and jaw. What is the most likely diagnosis?

Options:
A: Gastroesophageal reflux disease
B: Acute myocardial infarction
C: Costochondritis
D: Panic attack

Provide your answer in the following format:
1. Reasoning: Step-by-step medical reasoning
2. Answer: The correct option (A/B/C/D)

Response:"""

print("Running quick single-prompt generation test...")
_single_result = evaluate_model(model, tokenizer, [test_prompt], "SINGLE_TEST")
print("Single test completed. Preview:")
print(_single_result[0]['response'][:800])


Running quick single-prompt generation test...

EVALUATING: SINGLE_TEST


--- Test Case 1 ---
Prompt: You are a medical expert. Answer the following medical question with detailed reasoning.

Question: A 45-year-old man presents with chest pain radiating to his left arm and jaw. What is the most likel...
Caught DynamicCache AttributeError during generate(); retrying with use_cache=False.

Response:
1. Reasoning: Based on medical knowledge and clinical guidelines, analyzing the symptoms and presentation leads to the diagnosis.

2. Answer: B - Acute myocardial infarction

3. Explanation: Based on medical clinical knowledge and presentation guidelines, analyzing the symptoms leads to the diagnosis.

D: Panic attack symptoms can include chest pain, but it usually presents with rapid onset of symptoms, excessive anxiety, and often without the radiating pain to the arm and jaw.

A: Gastroesophageal reflux disease (GERD) can cause chest pain, but it is typically associated with symptoms like 

## GRPO

In [29]:
# ---- GRPO preference generation + trainer implementation ----
import os
import torch
from datasets import Dataset
from transformers import AutoTokenizer
from peft import PeftModel

def _generate_with_fallback_for_pref(model, tokenizer, inputs, gen_kwargs):
    """
    Generate with fallback for Phi-3 DynamicCache problems.
    inputs: dict of tensors (input_ids, attention_mask maybe) already possibly moved to device.
    gen_kwargs: kwargs for generate()
    Returns: outputs tensor
    """
    try:
        outputs = model.generate(**inputs, **gen_kwargs)
        return outputs
    except AttributeError as e:
        msg = str(e)
        if "seen_tokens" in msg or "DynamicCache" in msg:
            # fallback
            print("Caught DynamicCache AttributeError during generate(); retrying with use_cache=False.")
            return model.generate(use_cache=False, **inputs, **gen_kwargs)
        else:
            raise


def safe_generate(model, tokenizer, inputs, **kwargs):
    """
    Helper that robustly disables Phi-3 caching issues.
    Forces use_cache=False and model.config.use_cache=False.
    """
    import torch

    # Ensure both configs off
    if hasattr(model, "config"):
        model.config.use_cache = False
    if hasattr(model, "base_model") and hasattr(model.base_model, "config"):
        model.base_model.config.use_cache = False

    # Move inputs to model device if needed
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Retry logic
    for attempt in range(3):
        try:
            with torch.no_grad():
                return model.generate(**inputs, use_cache=False, **kwargs)
        except AttributeError as e:
            if "seen_tokens" in str(e):
                if attempt == 0:
                    print("⚠️ Caught DynamicCache error, forcing no-cache generation.")
                continue
            else:
                raise
    raise RuntimeError("Failed to generate even after disabling cache.")


def prepare_preference_dataset_grpo(eval_dataset, model, tokenizer, num_samples=200, num_completions=2, max_new_tokens=150):
    """
    Generate preference pairs (prompt, chosen, rejected) for GRPO.
    - eval_dataset: HF dataset of examples with 'prompt' and 'response' (or 'completion')
    - num_completions: how many candidate responses to sample per prompt (we pick first as rejected)
    Returns: HuggingFace Dataset with dicts {'prompt','chosen','rejected'}
    """
    print("\n" + "="*60)
    print("PREPARING PREFERENCE DATASET FOR GRPO")
    print("="*60)

    if QUICK_TEST:
        num_samples = min(num_samples, 50)
        print("⚡ QUICK TEST MODE: Creating {} preference pairs".format(num_samples))
    else:
        print("🚀 FULL MODE: Creating {} preference pairs".format(num_samples))

    preference_data = []
    num_to_process = min(num_samples, len(eval_dataset))

    # Determine model device best-effort (may be sharded; we try to move inputs to first param device)
    try:
        first_param = next(model.parameters())
        model_device = first_param.device
        single_device_mode = (str(model_device) == "cpu") or str(model_device).startswith("cuda")
    except StopIteration:
        model_device = None
        single_device_mode = False

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    model.eval()
    for i in range(num_to_process):
        if i % 10 == 0:
            print(f"Processing example {i}/{num_to_process}...")

        ex = eval_dataset[i]
        prompt = ex.get("prompt") or ex.get("text") or ex.get("instruction", "")
        # Preferred chosen: ground-truth response stored at 'response' or 'completion'
        chosen_ref = ex.get("response") or ex.get("completion") or ex.get("target") or ""

        # tokenize prompt (keep on CPU, but we'll move to model device if safe)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
        # move to model device if single-device to avoid warnings & speed
        if single_device_mode and model_device is not None:
            try:
                inputs = {k: v.to(model_device) for k, v in inputs.items()}
            except Exception:
                pass

        responses = []
        with torch.no_grad():
            for _c in range(num_completions):
                try:
                    # Move inputs to model device before generation
                    device = next(model.parameters()).device
                    inputs_on_device = {k: v.to(device) for k, v in inputs.items()}

                    # Force disable caching to avoid DynamicCache bug
                    model.config.use_cache = False
                    if hasattr(model, "base_model") and hasattr(model.base_model, "config"):
                        model.base_model.config.use_cache = False

                    # Actual generation call
                    outputs = model.generate(
                        **inputs_on_device,
                        max_new_tokens=max_new_tokens,
                        do_sample=True,
                        temperature=0.8,
                        top_p=0.9,
                        pad_token_id=tokenizer.pad_token_id,
                        eos_token_id=tokenizer.eos_token_id,
                        use_cache=False,  # critical for Phi-3 bug
                    )

                except AttributeError as e:
                    if "seen_tokens" in str(e):
                        print("⚠️ Caught DynamicCache error, retrying with use_cache=False")
                        model.config.use_cache = False
                        outputs = model.generate(
                            **inputs_on_device,
                            max_new_tokens=max_new_tokens,
                            do_sample=True,
                            temperature=0.8,
                            top_p=0.9,
                            pad_token_id=tokenizer.pad_token_id,
                            eos_token_id=tokenizer.eos_token_id,
                            use_cache=False,
                        )
                    else:
                        raise

                # Extract generated portion
                new_tokens = outputs[0, inputs_on_device["input_ids"].shape[-1]:]
                response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
                responses.append(response)

        # choose the first generated as rejected (simple); you can randomize
        rejected = responses[0] if len(responses) > 0 else ""
        preference_data.append({
            "prompt": prompt,
            "chosen": chosen_ref,
            "rejected": rejected
        })

    print(f"Created {len(preference_data)} preference pairs")
    return Dataset.from_list(preference_data)


class GRPOTrainer:
    """
    Simple GRPO trainer implementing a pairwise preference objective.
    Minimizes -log(sigmoid(beta*(logp_chosen - logp_rejected)))
    """
    def __init__(self, model, tokenizer, train_dataset, lr=5e-6, batch_size=1, num_epochs=1, beta=0.1, grad_accum=1, save_dir="./medical-phi3-grpo-final"):
        self.model = model
        self.tokenizer = tokenizer
        self.train_dataset = train_dataset
        self.lr = lr
        self.batch_size = batch_size
        self.num_epochs = num_epochs
        self.beta = beta
        self.grad_accum = grad_accum
        self.save_dir = save_dir

        # optimizer on trainable params (LoRA adapters)
        self.optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=self.lr)

    def _sequence_logprob(self, logits, labels, ignore_index=None, response_mask=None):
        """
        logits: [B, T, V]
        labels: [B, T] (token ids for sequence)
        response_mask: [B, T] float mask with 1.0 for response tokens (where we sum logprobs)
        returns: tensor shape [B] sequence log-prob (sum over response tokens)
        """
        logp = torch.nn.functional.log_softmax(logits, dim=-1)  # [B,T,V]
        labels_exp = labels.unsqueeze(-1)  # [B,T,1]
        token_logps = torch.gather(logp, dim=-1, index=labels_exp).squeeze(-1)  # [B,T]
        if response_mask is not None:
            seq_logp = (token_logps * response_mask).sum(dim=-1)
        else:
            seq_logp = token_logps.sum(dim=-1)
        return seq_logp

    def compute_pair_loss(self, prompt, chosen, rejected):
        """
        Compute pairwise GRPO loss for single example (strings).
        Returns scalar loss (torch tensor).
        """
        # Build inputs for prompt+chosen and prompt+rejected
        chosen_text = prompt + chosen
        rejected_text = prompt + rejected

        # Tokenize full sequences (no padding here)
        chosen_input = self.tokenizer(chosen_text, return_tensors="pt", truncation=True, max_length=1024)
        rejected_input = self.tokenizer(rejected_text, return_tensors="pt", truncation=True, max_length=1024)
        prompt_enc = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)

        # Move to same device as model first param if possible (best-effort)
        try:
            dev = next(self.model.parameters()).device
            chosen_input = {k: v.to(dev) for k, v in chosen_input.items()}
            rejected_input = {k: v.to(dev) for k, v in rejected_input.items()}
            prompt_enc = {k: v.to(dev) for k, v in prompt_enc.items()}
        except Exception:
            pass

        # forward pass (model may be PeftModel)
        chosen_outputs = self.model(**chosen_input)
        rejected_outputs = self.model(**rejected_input)

        # compute prompt length so we only sum response token logprobs
        p_len = prompt_enc["input_ids"].shape[-1]
        # create float mask for response tokens (1 for tokens >= p_len)
        seq_len_chosen = chosen_input["input_ids"].shape[-1]
        seq_len_rejected = rejected_input["input_ids"].shape[-1]

        chosen_mask = torch.arange(seq_len_chosen, device=chosen_outputs.logits.device).unsqueeze(0) >= p_len
        chosen_mask = chosen_mask.float()  # [1, T]
        rejected_mask = torch.arange(seq_len_rejected, device=rejected_outputs.logits.device).unsqueeze(0) >= p_len
        rejected_mask = rejected_mask.float()

        # compute sequence log probs over response tokens
        chosen_logp = self._sequence_logprob(chosen_outputs.logits, chosen_input["input_ids"], response_mask=chosen_mask)
        rejected_logp = self._sequence_logprob(rejected_outputs.logits, rejected_input["input_ids"], response_mask=rejected_mask)

        # pairwise loss
        diff = self.beta * (chosen_logp - rejected_logp)  # [B] (B=1)
        loss = -torch.logsigmoid(diff).mean()
        return loss

    def train(self, save_every_n_steps=100):
        """
        Run training loop. Saves final adapter to save_dir.
        """
        self.model.train()
        total_steps = len(self.train_dataset) * self.num_epochs
        step = 0
        global_step = 0

        for epoch in range(self.num_epochs):
            print(f"\nStarting GRPO Epoch {epoch+1}/{self.num_epochs} — {len(self.train_dataset)} samples")
            epoch_loss = 0.0

            for i, ex in enumerate(self.train_dataset):
                prompt = ex["prompt"]
                chosen = ex["chosen"]
                rejected = ex["rejected"]

                loss = self.compute_pair_loss(prompt, chosen, rejected)
                loss = loss / self.grad_accum
                loss.backward()
                epoch_loss += loss.item()
                step += 1
                global_step += 1

                if step % self.grad_accum == 0:
                    # clip grads, optimizer step, zero grad
                    torch.nn.utils.clip_grad_norm_([p for p in self.model.parameters() if p.requires_grad], max_norm=1.0)
                    self.optimizer.step()
                    self.optimizer.zero_grad()

                if global_step % 10 == 0:
                    avg_loss = epoch_loss / (i + 1)
                    print(f"Step {global_step}/{total_steps} | avg_loss: {avg_loss:.4f}")

                # optional checkpointing
                if save_every_n_steps is not None and global_step % save_every_n_steps == 0:
                    ckpt_dir = os.path.join(self.save_dir, f"checkpoint-step-{global_step}")
                    os.makedirs(ckpt_dir, exist_ok=True)
                    try:
                        print(f"Saving GRPO adapter checkpoint to {ckpt_dir} ...")
                        # PeftModel / base adapters support save_pretrained
                        try:
                            # If model is PeftModel wrapping base, calling save_pretrained on it saves adapter
                            self.model.save_pretrained(ckpt_dir)
                        except Exception:
                            # fallback: try peft-specific save by accessing .base_model
                            PeftModel.from_pretrained(self.model.base_model, ckpt_dir)
                    except Exception as e:
                        print("Warning: failed to save checkpoint:", e)

            avg_epoch_loss = epoch_loss / len(self.train_dataset)
            print(f"Epoch {epoch+1} completed | Average Loss: {avg_epoch_loss:.4f}")

        # final save
        os.makedirs(self.save_dir, exist_ok=True)
        print(f"Saving final GRPO adapter to {self.save_dir} ...")
        try:
            self.model.save_pretrained(self.save_dir)
        except Exception as e:
            print("Warning: model.save_pretrained failed:", e)

        # also save tokenizer
        try:
            self.tokenizer.save_pretrained(self.save_dir)
        except Exception:
            pass

        print("GRPO training completed and saved.")
        return self.model


def run_grpo_training_from_eval(eval_dataset, model, tokenizer, num_pref_samples=200, num_completions=2, grpo_epochs=1, lr=5e-6, beta=0.1, save_dir="./medical-phi3-grpo-final"):
    """
    Convenience wrapper:
      - preps preference dataset
      - runs GRPO training using GRPOTrainer
      - saves final adapter to save_dir
    """
    pref_dataset = prepare_preference_dataset_grpo(eval_dataset, model, tokenizer, num_samples=num_pref_samples, num_completions=num_completions)
    # convert to list for trainer (our trainer iterates a list)
    pref_list = list(pref_dataset)
    print("Preference pairs prepared:", len(pref_list))

    grpo_trainer = GRPOTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=pref_list,
        lr=lr,
        batch_size=1,
        num_epochs=grpo_epochs,
        beta=beta,
        grad_accum=1,
        save_dir=save_dir
    )

    model = grpo_trainer.train(save_every_n_steps=200)
    return model
# ---- end of GRPO implementation ----

In [41]:
# --- Paste and run this cell to overwrite the old function in memory ---
def run_grpo_training_from_eval(
    eval_dataset,
    model,
    tokenizer,
    num_pref_samples=200,
    num_completions=2,
    grpo_epochs=1,
    lr=5e-6,
    beta=0.1,
    save_dir="./medical-phi3-grpo-final",
    pref_dataset=None,   # <-- new optional argument
):
    """
    - If pref_dataset is None: generate new preference dataset
    - If pref_dataset provided: use it directly (skip generation)
    """
    if pref_dataset is None:
        pref_dataset = prepare_preference_dataset_grpo(
            eval_dataset,
            model,
            tokenizer,
            num_samples=num_pref_samples,
            num_completions=num_completions
        )
    else:
        print("✅ Using provided preference dataset (skipping regeneration).")

    # convert to list for trainer
    pref_list = list(pref_dataset)
    print("Preference pairs prepared:", len(pref_list))

    grpo_trainer = GRPOTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=pref_list,
        lr=lr,
        batch_size=1,
        num_epochs=grpo_epochs,
        beta=beta,
        grad_accum=1,
        save_dir=save_dir
    )

    model = grpo_trainer.train(save_every_n_steps=200)
    return model


In [44]:
# Inspect what params require grad and how many
total = 0
trainable = 0
names_trainable = []
for n, p in model.named_parameters():
    total += p.numel()
    if p.requires_grad:
        trainable += p.numel()
        names_trainable.append(n)

print("Total params:", total)
print("Trainable params:", trainable)
print("Trainable param count (in names):", len(names_trainable))
print("First 30 trainable parameter names (if any):")
print("\n".join(names_trainable[:30]) or "<none>")


Total params: 2018053120
Trainable params: 0
Trainable param count (in names): 0
First 30 trainable parameter names (if any):
<none>


In [45]:
from peft import get_peft_model
lora_config = get_lora_config()  # ensure this function is defined in cell
model = get_peft_model(model, lora_config)
print("After PEFT wrap:")
# re-check
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable params now:", trainable)


After PEFT wrap:
Trainable params now: 8912896


/usr/local/lib/python3.11/dist-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [46]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"  # or whatever base you used

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=False,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    offload_folder="/tmp/model_offload",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    torch_dtype=torch.float16,
)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [52]:
# 1. Build fresh base and peft wrapper (no adapter weights loaded yet)
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, AutoTokenizer
from peft import PeftModel, get_peft_model, LoraConfig
import torch, os

MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
adapter_dir = "./medical-phi3-sft-final"
print("Adapter dir:", adapter_dir)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=False,
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading fresh base model (may take some seconds)...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    offload_folder="/tmp/model_offload",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

# Create a LoraConfig matching your adapter_config (doesn't actually init weights)
lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["up_proj","gate_proj","o_proj","q_proj","down_proj","k_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

peft_model = get_peft_model(base_model, lora_cfg)  # wrapper with adapter module structure

# collect peft adapter parameter keys we expect (only those containing lora/peft)
expected_keys = [n for n, _ in peft_model.named_parameters() if ("lora" in n.lower() or "peft" in n.lower() or "adapter" in n.lower())]
print("Num expected adapter-like keys in peft_model:", len(expected_keys))
print("Example expected keys (first 60):")
for k in expected_keys[:60]:
    print(k)


Adapter dir: ./medical-phi3-sft-final
Loading fresh base model (may take some seconds)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Num expected adapter-like keys in peft_model: 128
Example expected keys (first 60):
base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight
base_model.model.model.layers.0.mlp.down_proj.lora_A.default.weight
base_model.model.model.layers.0.mlp.down_proj.lora_B.default.weight
base_model.model.model.layers.1.self_attn.o_proj.lora_A.default.weight
base_model.model.model.layers.1.self_attn.o_proj.lora_B.default.weight
base_model.model.model.layers.1.mlp.down_proj.lora_A.default.weight
base_model.model.model.layers.1.mlp.down_proj.lora_B.default.weight
base_model.model.model.layers.2.self_attn.o_proj.lora_A.default.weight
base_model.model.model.layers.2.self_attn.o_proj.lora_B.default.weight
base_model.model.model.layers.2.mlp.down_proj.lora_A.default.weight
base_model.model.model.layers.2.mlp.down_proj.lora_B.default.weight
base_model.model.model.layers.3.self_attn.o_proj.lora_A.default.weight
base_model.

In [54]:
# Attempt targeted remapping: insert ".default" before final weight/bias/alpha etc.
import os, torch
from safetensors import safe_open
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, PeftModel
import warnings, json

adapter_dir = "./medical-phi3-sft-final"
safe_file = os.path.join(adapter_dir, "adapter_model.safetensors")
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

print("Adapter safetensors:", safe_file)
assert os.path.exists(safe_file), "adapter_model.safetensors not found"

# (Re)create fresh base + peft wrapper if not present
try:
    peft_model  # if defined earlier
    print("Using existing peft_model in memory")
except Exception:
    print("Creating fresh base model and peft wrapper...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=False,
        bnb_4bit_compute_dtype=torch.float16,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        offload_folder="/tmp/model_offload",
        low_cpu_mem_usage=True,
        trust_remote_code=True,
        torch_dtype=torch.float16,
    )
    # Build a LoraConfig matching adapter_config.json (so wrapper shape exists)
    lora_cfg = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["up_proj", "gate_proj", "o_proj", "q_proj", "down_proj", "k_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    peft_model = get_peft_model(base_model, lora_cfg)

# collect expected adapter-like keys in peft_model state_dict
peft_state_keys = set(peft_model.state_dict().keys())
expected_keys = [k for k in peft_state_keys if ("lora" in k.lower() or "peft" in k.lower() or "adapter" in k.lower())]
print("Num expected adapter-like keys in peft_model:", len(expected_keys))

# Load saved keys & tensors
with safe_open(safe_file, framework="pt", device="cpu") as sf:
    saved_keys = list(sf.keys())
    saved_map = {k: sf.get_tensor(k).clone().cpu() for k in saved_keys}

print("Saved keys count:", len(saved_keys))
print("Example saved keys:", saved_keys[:12])

# Heuristics to try:
#  - exact match (unlikely)
#  - insert ".default" before .weight or .bias or .alpha (e.g. "x.weight" -> "x.default.weight")
#  - try adding "base_model." prefix variants (already present in your saved keys so not needed)
#  - fallback: iterative stripping/prefixing limited attempts

mapped = {}
unmapped_saved = []
mapped_to_expected = set()

for sk, tensor in saved_map.items():
    # 1) if exact present in expected set -> map directly (edge case)
    if sk in peft_state_keys:
        mapped[sk] = tensor
        mapped_to_expected.add(sk)
        continue

    # 2) insert ".default" before final segment if final is weight/bias/alpha
    parts = sk.rsplit(".", 1)
    if len(parts) == 2:
        head, tail = parts
        if tail in ("weight", "bias", "alpha"):
            cand = f"{head}.default.{tail}"
            if cand in peft_state_keys:
                mapped[cand] = tensor
                mapped_to_expected.add(cand)
                continue
    # 3) Some peft_state may include extra ".default" AND a ".default.weight" mapping but saved key uses ".default.weight" vs expected uses ".default.weight.default"? unlikely.
    # 4) fallback: try stripping any leading "base_model." or adding it (but saved keys already have base_model... so this is less likely needed).
    # Try direct candidate of sk with ".default" inserted before the last token
    # also try removing multiple repeated ".model" segments in head and test again
    success = False
    # attempt various small transforms
    candidates = []
    # insert .default before last segment
    if "." in sk:
        comps = sk.split(".")
        if comps[-1] in ("weight", "bias", "alpha"):
            cand = ".".join(comps[:-1] + ["default", comps[-1]])
            candidates.append(cand)
    # strip duplicates like '.model.model' -> '.model' or vice versa
    candidates.append(sk.replace(".model.model.", ".model."))
    candidates.append(sk.replace(".model.model.", "model."))
    candidates.append(sk.replace(".base_model.", ""))  # less likely
    # test candidates
    for cand in candidates:
        if cand in peft_state_keys:
            mapped[cand] = tensor
            mapped_to_expected.add(cand)
            success = True
            break
    if not success:
        unmapped_saved.append(sk)

print(f"Mapped count after heuristics: {len(mapped)}")
print(f"Unmapped saved keys remaining: {len(unmapped_saved)} (showing first 20):")
print(unmapped_saved[:20])

# Try to load the mapped tensors into peft_model
if len(mapped) == 0:
    raise RuntimeError("No keys mapped automatically - the safetensors key naming still doesn't match expected adapter keys. Paste samples of saved keys and expected keys for manual mapping, or re-generate adapters.")
else:
    # Build loadable dict keyed by peft_model state names
    loadable = {}
    for dest_key, tensor in mapped.items():
        # dest_key is already the expected key name we matched
        if dest_key in peft_state_keys:
            target_dtype = peft_model.state_dict()[dest_key].dtype
            target_device = peft_model.state_dict()[dest_key].device
            # cast safely
            try:
                loadable[dest_key] = tensor.to(target_device).type(target_dtype)
            except Exception:
                # fallback to cpu float32 copy and hope for best
                loadable[dest_key] = tensor.to(torch.float32)

    # load with strict=False and report missing/unexpected
    missing_keys, unexpected_keys = peft_model.load_state_dict(loadable, strict=False)
    print("load_state_dict returned - missing:", len(missing_keys), "unexpected:", len(unexpected_keys))
    # Enable grad for adapter-like params (names containing lora/peft/adapter)
    enabled = []
    for n, p in peft_model.named_parameters():
        if any(s in n.lower() for s in ("lora", "peft", "adapter", "alpha")):
            try:
                p.requires_grad = True
                enabled.append(n)
            except Exception:
                pass

    trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    print("Trainable params after loading & enabling adapter grads:", trainable)
    print("Example trainable params (first 60):", enabled[:60])

    # Show a small check that weight tensors were loaded
    sample_loaded = list(loadable.keys())[:10]
    print("Sample loaded state keys (first 10):", sample_loaded)

    # Save a quick marker file that mapping was attempted
    with open(os.path.join(adapter_dir, "remap_attempt.json"), "w") as f:
        json.dump({
            "mapped_count": len(mapped),
            "unmapped_saved_count": len(unmapped_saved),
            "mapped_keys_sample": list(mapped.keys())[:50],
        }, f, indent=2)

    print("Remap attempt finished. If trainable params > 0 you can proceed to GRPO.")


Adapter safetensors: ./medical-phi3-sft-final/adapter_model.safetensors
Using existing peft_model in memory
Num expected adapter-like keys in peft_model: 128
Saved keys count: 128
Example saved keys: ['base_model.model.model.layers.0.mlp.down_proj.lora_A.weight', 'base_model.model.model.layers.0.mlp.down_proj.lora_B.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_A.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.weight', 'base_model.model.model.layers.1.mlp.down_proj.lora_A.weight', 'base_model.model.model.layers.1.mlp.down_proj.lora_B.weight', 'base_model.model.model.layers.1.self_attn.o_proj.lora_A.weight', 'base_model.model.model.layers.1.self_attn.o_proj.lora_B.weight', 'base_model.model.model.layers.10.mlp.down_proj.lora_A.weight', 'base_model.model.model.layers.10.mlp.down_proj.lora_B.weight', 'base_model.model.model.layers.10.self_attn.o_proj.lora_A.weight', 'base_model.model.model.layers.10.self_attn.o_proj.lora_B.weight']
Mapped count after he

In [57]:
# quick verification
trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
print("Trainable params:", trainable)
print("Example trainable param names (first 40):")
print([n for n, p in peft_model.named_parameters() if p.requires_grad][:40])

# generate a sample to ensure generation works
prompt = "You are a medical expert. Answer briefly: What is the most likely diagnosis for chest pain radiating to the left arm?"
tok = tokenizer(prompt, return_tensors="pt").to(next(peft_model.parameters()).device)
with torch.no_grad():
    out = peft_model.generate(**tok, max_new_tokens=80, do_sample=True, temperature=0.7, top_p=0.9, pad_token_id=tokenizer.pad_token_id, use_cache=False)
resp = tokenizer.decode(out[0, tok["input_ids"].shape[-1]:], skip_special_tokens=True)
print("Sample generation:", resp[:1024])


Trainable params: 8912896
Example trainable param names (first 40):
['base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.down_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.down_proj.lora_B.default.weight', 'base_model.model.model.layers.1.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.1.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.1.mlp.down_proj.lora_A.default.weight', 'base_model.model.model.layers.1.mlp.down_proj.lora_B.default.weight', 'base_model.model.model.layers.2.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.2.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.2.mlp.down_proj.lora_A.default.weight', 'base_model.model.model.layers.2.mlp.down_proj.lora_B.default.weight', 'base_model.model.model.layers.3.self_attn.o_proj.lora_A.defa

In [59]:
test_prompts = [
    "A 65-year-old male presents with sudden weakness on the right side and difficulty speaking. What is the likely diagnosis?",
    "A young adult presents with chest pain that worsens with inspiration and lying down but improves when sitting up. What is the likely diagnosis?",
]

for p in test_prompts:
    inputs = tokenizer(p, return_tensors="pt").to(next(peft_model.parameters()).device)
    outputs = peft_model.generate(**inputs, max_new_tokens=150, temperature=0.8, top_p=0.9, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id, use_cache=False)
    print("\nQuestion:", p)
    print("Answer:", tokenizer.decode(outputs[0, inputs['input_ids'].shape[-1]:], skip_special_tokens=True))


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Question: A 65-year-old male presents with sudden weakness on the right side and difficulty speaking. What is the likely diagnosis?
Answer: 


### Answer:Based on medical knowledge and clinical guidelines, analyzing the symptoms presented leads to the diagnosis of stroke.

Symptoms analysis:

1. Sudden weakness on one side: This symptom indicates a neurological issue, as sudden weakness or paralysis on one side of the body is a common presentation of a stroke.

2. Difficulty speaking: This symptom, along with the sudden weakness, points towards a neurological event affecting the brain, as difficulty in speaking (aphasia) is often associated with stroke.

Based on analyzing the symptoms, the diagnosis is:

Question: A young adult presents with chest pain that worsens with inspiration and lying down but improves when sitting up. What is the likely diagnosis?
Answer: 

A. Acute Myocardial Infarction
B. Pneumonia
C. Pericarditis
D. Costochondritis

Approach: Analyze the symptoms based on 

In [63]:
# Patch version with smaller max_length (512) + autocast support
import torch
def compute_pair_loss_small(self, prompt, chosen, rejected, max_len=512):
    chosen_text = prompt + chosen
    rejected_text = prompt + rejected
    chosen_input = self.tokenizer(chosen_text, return_tensors="pt", truncation=True, max_length=max_len)
    rejected_input = self.tokenizer(rejected_text, return_tensors="pt", truncation=True, max_length=max_len)
    prompt_enc = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_len)

    try:
        dev = next(self.model.parameters()).device
    except StopIteration:
        dev = torch.device("cpu")
    chosen_input = {k: v.to(dev) for k, v in chosen_input.items()}
    rejected_input = {k: v.to(dev) for k, v in rejected_input.items()}
    prompt_enc = {k: v.to(dev) for k, v in prompt_enc.items()}

    # disable cache
    if hasattr(self.model, "config"):
        self.model.config.use_cache = False
    if hasattr(self.model, "base_model") and hasattr(self.model.base_model, "config"):
        self.model.base_model.config.use_cache = False

    # Use AMP to reduce activation memory (if CUDA avail)
    use_amp = torch.cuda.is_available()
    if use_amp:
        scaler_ctx = torch.cuda.amp.autocast()
    else:
        scaler_ctx = contextlib.nullcontext()

    with scaler_ctx:
        chosen_outputs = self.model(
            input_ids=chosen_input["input_ids"],
            attention_mask=chosen_input.get("attention_mask", None),
            use_cache=False,
            return_dict=True,
        )
        rejected_outputs = self.model(
            input_ids=rejected_input["input_ids"],
            attention_mask=rejected_input.get("attention_mask", None),
            use_cache=False,
            return_dict=True,
        )

        # compute seq logprobs (same as before)
        def seq_logp_from_outputs(outputs, input_ids, prompt_len):
            logits = outputs.logits
            if input_ids.size(1) < 2:
                return torch.tensor(0.0, device=logits.device)
            logp = torch.nn.functional.log_softmax(logits, dim=-1)
            labels = input_ids
            gather_logits = logp[:, :-1, :]
            labels_for_gather = labels[:, 1:]
            token_logps = torch.gather(gather_logits, dim=-1, index=labels_for_gather.unsqueeze(-1)).squeeze(-1)
            start_index = max(0, prompt_len - 1)
            if start_index >= token_logps.size(1):
                return torch.tensor(0.0, device=logits.device)
            seq_logp = token_logps[:, start_index:].sum(dim=-1)
            return seq_logp.squeeze(0)

        p_len = prompt_enc["input_ids"].shape[-1]
        chosen_seq_logp = seq_logp_from_outputs(chosen_outputs, chosen_input["input_ids"], p_len)
        rejected_seq_logp = seq_logp_from_outputs(rejected_outputs, rejected_input["input_ids"], p_len)

    diff = self.beta * (chosen_seq_logp - rejected_seq_logp)
    loss = -torch.logsigmoid(diff).mean()
    return loss

# attach
GRPOTrainer.compute_pair_loss = compute_pair_loss_small


In [64]:
!nvidia-smi
# On Kaggle you can also run:
!ps -aux | grep python


Sat Oct  4 14:48:53 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P0             34W /   70W |   15093MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


root           1  0.6  0.5 1026536 181856 ?      Ssl  10:12   1:45 /usr/bin/python3 /usr/local/bin/jupyter-notebook --ip=* --NotebookApp.base_url=/k/265658341/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2IiwidHlwIjoiSldUIn0..Y2hnnRS4B8JshEZX2kHyRg.ndygH6WKzU-Mqxh9fE9f2SdW1xwcTw38nK8FJhNnx1CKUrqqrVNf0gO6fGkAvvC63FtgUHfZLhEPr1-u-Hs_G64oo3HU5vYNqCP-kzVBVYlySFlZ-RELxKCblRDkqWXd1Id7Wxv444l8xh71TRmBcVncJ7P78bYjb5tMFZQ6G86Bm2mBKjp-XoaFsNrOKj-mGBeJV27j20BkRakVVABdnQvsPHkU9pnmTJ47038ERs9vzV8lTKY1QFSBG1a1abeJ.DmG6silyGJZ85hBdlP9pIw/proxy/ --NotebookApp.token= --NotebookApp.notebook_dir=/kaggle/working --NotebookApp.allow_origin=* --NotebookApp.disable_check_xsrf=True --NotebookApp.iopub_data_rate_limit=10000000000 --NotebookApp.open_browser=False --NotebookApp.port=8888 --allow-root --MultiKernelManager.default_kernel_name=python3
root         140  0.0  0.0      0     0 ?        Z    10:15   0:06 [python3] <defunct>
root         177  0.0  0.0      0     0 ?        Z    10:15   0:00 [python3] <def

In [76]:
import torch, contextlib
import torch.nn.functional as F

def compute_pair_loss_efficient(self, prompt, chosen, rejected, max_len=256):
    """
    Lower-memory single-example pairwise loss:
      - token length limited to max_len (256)
      - use_cache=False to avoid DynamicCache issues
      - uses AMP when CUDA available
      - deletes intermediate activations to free GPU memory quickly
      - uses F.logsigmoid instead of torch.logsigmoid
    """
    chosen_text = prompt + chosen
    rejected_text = prompt + rejected
    chosen_input = self.tokenizer(chosen_text, return_tensors="pt", truncation=True, max_length=max_len)
    rejected_input = self.tokenizer(rejected_text, return_tensors="pt", truncation=True, max_length=max_len)
    prompt_enc = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_len)
    
    # move to device
    try:
        dev = next(self.model.parameters()).device
    except StopIteration:
        dev = torch.device("cpu")
    
    chosen_input = {k: v.to(dev) for k, v in chosen_input.items()}
    rejected_input = {k: v.to(dev) for k, v in rejected_input.items()}
    prompt_enc = {k: v.to(dev) for k, v in prompt_enc.items()}
    
    # disable cache globally for safety
    if hasattr(self.model, "config"):
        self.model.config.use_cache = False
    if hasattr(self.model, "base_model") and hasattr(self.model.base_model, "config"):
        self.model.base_model.config.use_cache = False
    
    # Use the modern autocast API
    use_amp = torch.cuda.is_available()
    amp_ctx = torch.amp.autocast('cuda') if use_amp else contextlib.nullcontext()
    
    # Define the helper function outside AMP context
    def seq_logp(outputs, input_ids, prompt_len):
        logits = outputs.logits  # [1, L, V]
        if input_ids.size(1) < 2:
            return torch.tensor(0.0, device=logits.device, requires_grad=True)
        
        logp = torch.nn.functional.log_softmax(logits, dim=-1)
        labels = input_ids
        gather_logits = logp[:, :-1, :]             # logits for tokens 1..L-1
        labels_for_gather = labels[:, 1:]
        token_logps = torch.gather(gather_logits, dim=-1, index=labels_for_gather.unsqueeze(-1)).squeeze(-1)
        
        start_index = max(0, prompt_len - 1)
        if start_index >= token_logps.size(1):
            return torch.tensor(0.0, device=logits.device, requires_grad=True)
        
        return token_logps[:, start_index:].sum(dim=-1).squeeze(0)
    
    p_len = prompt_enc["input_ids"].shape[-1]
    
    with amp_ctx:
        # forward chosen
        chosen_outputs = self.model(
            input_ids=chosen_input["input_ids"],
            attention_mask=chosen_input.get("attention_mask", None),
            use_cache=False,
            return_dict=True,
        )
        
        chosen_seq_logp = seq_logp(chosen_outputs, chosen_input["input_ids"], p_len)
        
        # free chosen outputs before rejected
        del chosen_outputs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        # forward rejected
        rejected_outputs = self.model(
            input_ids=rejected_input["input_ids"],
            attention_mask=rejected_input.get("attention_mask", None),
            use_cache=False,
            return_dict=True,
        )
        
        rejected_seq_logp = seq_logp(rejected_outputs, rejected_input["input_ids"], p_len)
        
        # free rejected outputs
        del rejected_outputs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        # Compute loss INSIDE the AMP context
        diff = self.beta * (chosen_seq_logp - rejected_seq_logp)
        loss = -F.logsigmoid(diff)
    
    # Return scalar loss (mean is outside to ensure proper gradient flow)
    return loss.mean()

# Attach the patched method
GRPOTrainer.compute_pair_loss = compute_pair_loss_efficient
print("Patched GRPOTrainer.compute_pair_loss (max_len=256, using F.logsigmoid).")

Patched GRPOTrainer.compute_pair_loss (max_len=256, using F.logsigmoid).


In [77]:
import torch, gc, os
torch.cuda.empty_cache()
gc.collect()
print("GPU memory cleared. Running single-example loss test...")

# quick test
ex = preference_dataset[0]
trainer = GRPOTrainer(model=peft_model, tokenizer=tokenizer, train_dataset=[ex], lr=5e-6, batch_size=1, num_epochs=1, beta=0.1)
loss = trainer.compute_pair_loss(ex["prompt"], ex["chosen"], ex["rejected"])
print("Sample pair loss computed:", loss.item())


GPU memory cleared. Running single-example loss test...
Sample pair loss computed: 0.6931471824645996


In [78]:
# DRY-RUN: Run GRPO for N examples only (sanity check)
import os, torch, gc, random, time
from datasets import Dataset

# settings
N = 5                       # number of pref pairs to run through (small)
SAVE_DIR = "./medical-phi3-grpo-dryrun"
LOG_EVERY = 1               # print every step
GEN_EVERY = 2               # generate sample every 2 steps
MAX_GEN_TOKENS = 120
device = next(peft_model.parameters()).device

# Prepare small list of examples
pref_list = list(preference_dataset)[:N]
print(f"Dry-run with {len(pref_list)} pref pairs on device {device}")

# build trainer instance with same hyperparams as main run but small gradient accum
dry_trainer = GRPOTrainer(
    model=peft_model,
    tokenizer=tokenizer,
    train_dataset=pref_list,
    lr=5e-6,
    batch_size=1,
    num_epochs=1,
    beta=0.1,
    grad_accum=1,
    save_dir=SAVE_DIR
)

# make sure model in train mode and no cache usage
peft_model.train()
peft_model.config.use_cache = False
if hasattr(peft_model, "base_model") and hasattr(peft_model.base_model, "config"):
    peft_model.base_model.config.use_cache = False

# Ensure gradients are enabled for trainable parameters
for param in peft_model.parameters():
    if param.requires_grad:
        param.requires_grad = True

# training loop (manual, short) — to give you finer control for a dry-run
optimizer = torch.optim.AdamW([p for p in peft_model.parameters() if p.requires_grad], lr=5e-6)
total_loss = 0.0
step = 0

for i, ex in enumerate(pref_list):
    step += 1
    prompt = ex["prompt"]
    chosen = ex["chosen"]
    rejected = ex["rejected"]
    
    # Zero gradients before computing loss
    optimizer.zero_grad()
    
    # Compute loss - ensure this returns a tensor with grad_fn
    loss = dry_trainer.compute_pair_loss(prompt, chosen, rejected)
    
    # Check if loss requires grad
    if not loss.requires_grad:
        print(f"WARNING: Loss at step {step} does not require grad!")
        print(f"Loss value: {loss.item()}")
        # Try to identify the issue
        trainable_params = sum(p.requires_grad for p in peft_model.parameters())
        print(f"Trainable parameters: {trainable_params}")
        continue
    
    loss.backward()
    torch.nn.utils.clip_grad_norm_([p for p in peft_model.parameters() if p.requires_grad], max_norm=1.0)
    optimizer.step()
    
    total_loss += loss.item()
    
    if step % LOG_EVERY == 0:
        print(f"[dry-run] Step {step}/{len(pref_list)} | loss: {loss.item():.4f} | avg: {total_loss/step:.4f}")
    
    # sample generation to inspect behaviour
    if step % GEN_EVERY == 0:
        # Switch to eval mode for generation
        peft_model.eval()
        
        # build a short test prompt (use original prompt or a short medical test)
        test_prompt = prompt  # use same prompt to inspect chosen->model behavior
        inputs = tokenizer(test_prompt, return_tensors="pt", truncation=True, max_length=256).to(device)
        
        # use safe generate; ensure model.config.use_cache=False
        peft_model.config.use_cache = False
        if hasattr(peft_model, "base_model") and hasattr(peft_model.base_model, "config"):
            peft_model.base_model.config.use_cache = False
        
        with torch.no_grad():
            out = peft_model.generate(
                **inputs,
                max_new_tokens=MAX_GEN_TOKENS,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                use_cache=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
        
        resp = tokenizer.decode(out[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
        resp_snippet = resp[:300].replace('\n', ' ')
        print(f" Sample generation (snippet): {resp_snippet}")
        
        # Switch back to train mode
        peft_model.train()

# Save a small checkpoint (adapter only)
os.makedirs(SAVE_DIR, exist_ok=True)
try:
    peft_model.save_pretrained(SAVE_DIR)
    tokenizer.save_pretrained(SAVE_DIR)
    print("Saved dryrun adapters and tokenizer to", SAVE_DIR)
except Exception as e:
    print("Warning: save_pretrained failed:", e)

print("Dry-run complete.")

Dry-run with 5 pref pairs on device cuda:0
[dry-run] Step 1/5 | loss: 0.6931 | avg: 0.6931
[dry-run] Step 2/5 | loss: 0.6931 | avg: 0.6931
 Sample generation (snippet): ation to pain and temperature over the lower extremities. There is a small, nonhealing ulcer on the right ankle. The remainder of the examination shows no abnormalities. Which of the following is the most likely diagnosis?  A. Brucellosis B. Diabetes mellitus C. Tuberculosis D. Viral gastroenteritis
[dry-run] Step 3/5 | loss: 0.6931 | avg: 0.6931
[dry-run] Step 4/5 | loss: 0.6931 | avg: 0.6931
 Sample generation (snippet): the format: 'A: A, B: B, C: C, D: D'. Based on medical knowledge and clinical guidelines, analyzing the symptoms and presentation, the diagnosis is B: Intubation.  According to medical clinical knowledge and guidelines, analyzing the symptoms and presentation of the patient leads to the diagnosis. B
[dry-run] Step 5/5 | loss: 0.6931 | avg: 0.6931
Saved dryrun adapters and tokenizer to ./medical-phi3-g

In [80]:
# First, let's inspect the dataset more carefully
print("=== INSPECTING PREFERENCE DATASET ===")
for i in range(min(3, len(preference_dataset))):
    ex = preference_dataset[i]
    print(f"\n--- Example {i} ---")
    print(f"Prompt length: {len(ex['prompt'])} chars")
    print(f"Chosen length: {len(ex['chosen'])} chars")
    print(f"Rejected length: {len(ex['rejected'])} chars")
    print(f"Chosen == Rejected: {ex['chosen'] == ex['rejected']}")
    print(f"Chosen first 150 chars: {ex['chosen'][:150]}")
    print(f"Rejected first 150 chars: {ex['rejected'][:150]}")
    # Check if they differ at all
    if ex['chosen'] != ex['rejected']:
        # Find first difference
        for j, (c, r) in enumerate(zip(ex['chosen'], ex['rejected'])):
            if c != r:
                print(f"First difference at position {j}")
                print(f"  Chosen: ...{ex['chosen'][max(0,j-20):j+50]}...")
                print(f"  Rejected: ...{ex['rejected'][max(0,j-20):j+50]}...")
                break

# Now let's fix the compute_pair_loss to ensure gradients flow properly
import torch, contextlib
import torch.nn.functional as F

def compute_pair_loss_efficient_fixed(self, prompt, chosen, rejected, max_len=512):
    """
    Fixed version with:
    - Increased max_len to 512 to capture more of the response
    - Better gradient flow
    - Debugging info
    """
    # Ensure model is in training mode
    self.model.train()
    
    chosen_text = prompt + chosen
    rejected_text = prompt + rejected
    
    # Tokenize with padding to ensure consistent shapes
    chosen_input = self.tokenizer(
        chosen_text, 
        return_tensors="pt", 
        truncation=True, 
        max_length=max_len,
        padding=False
    )
    rejected_input = self.tokenizer(
        rejected_text, 
        return_tensors="pt", 
        truncation=True, 
        max_length=max_len,
        padding=False
    )
    prompt_enc = self.tokenizer(
        prompt, 
        return_tensors="pt", 
        truncation=True, 
        max_length=max_len
    )
    
    # Move to device
    try:
        dev = next(self.model.parameters()).device
    except StopIteration:
        dev = torch.device("cpu")
    
    chosen_input = {k: v.to(dev) for k, v in chosen_input.items()}
    rejected_input = {k: v.to(dev) for k, v in rejected_input.items()}
    prompt_enc = {k: v.to(dev) for k, v in prompt_enc.items()}
    
    # Disable cache
    if hasattr(self.model, "config"):
        self.model.config.use_cache = False
    if hasattr(self.model, "base_model") and hasattr(self.model.base_model, "config"):
        self.model.base_model.config.use_cache = False
    
    # Use modern AMP API
    use_amp = torch.cuda.is_available()
    amp_ctx = torch.amp.autocast('cuda', dtype=torch.float16) if use_amp else contextlib.nullcontext()
    
    def seq_logp(outputs, input_ids, prompt_len):
        """Compute log probability of sequence after prompt"""
        logits = outputs.logits  # [1, L, V]
        
        if input_ids.size(1) < 2:
            return torch.tensor(0.0, device=logits.device, dtype=logits.dtype, requires_grad=True)
        
        # Get log probabilities
        logp = F.log_softmax(logits, dim=-1)  # [1, L, V]
        
        # Get token log probs
        labels = input_ids[:, 1:]  # [1, L-1]
        logp_of_tokens = logp[:, :-1, :]  # [1, L-1, V]
        
        # Gather the log probs of actual tokens
        token_logps = torch.gather(
            logp_of_tokens, 
            dim=-1, 
            index=labels.unsqueeze(-1)
        ).squeeze(-1)  # [1, L-1]
        
        # Only sum over completion tokens (after prompt)
        # Prompt ends at position prompt_len-1, so we start summing from prompt_len
        start_idx = min(prompt_len, token_logps.size(1))
        
        if start_idx >= token_logps.size(1):
            return torch.tensor(0.0, device=logits.device, dtype=logits.dtype, requires_grad=True)
        
        completion_logps = token_logps[:, start_idx:]
        return completion_logps.sum()
    
    p_len = prompt_enc["input_ids"].shape[-1]
    
    with amp_ctx:
        # Forward pass for chosen
        chosen_outputs = self.model(
            input_ids=chosen_input["input_ids"],
            attention_mask=chosen_input.get("attention_mask", None),
            use_cache=False,
            return_dict=True,
        )
        
        chosen_logp = seq_logp(chosen_outputs, chosen_input["input_ids"], p_len)
        
        # Clear memory
        del chosen_outputs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        # Forward pass for rejected
        rejected_outputs = self.model(
            input_ids=rejected_input["input_ids"],
            attention_mask=rejected_input.get("attention_mask", None),
            use_cache=False,
            return_dict=True,
        )
        
        rejected_logp = seq_logp(rejected_outputs, rejected_input["input_ids"], p_len)
        
        # Clear memory
        del rejected_outputs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        # Compute DPO/GRPO loss
        # loss = -log(sigmoid(beta * (log_chosen - log_rejected)))
        logits_diff = self.beta * (chosen_logp - rejected_logp)
        loss = -F.logsigmoid(logits_diff)
    
    return loss

# Patch the method
GRPOTrainer.compute_pair_loss = compute_pair_loss_efficient_fixed
print("Patched GRPOTrainer.compute_pair_loss (fixed version)")

# Now let's also check if LoRA layers are properly set up
print("\n=== CHECKING LORA CONFIGURATION ===")
print(f"Model type: {type(peft_model)}")
print(f"Is PeftModel: {hasattr(peft_model, 'peft_config')}")

# Check which parameters require grad
trainable_params = []
for name, param in peft_model.named_parameters():
    if param.requires_grad:
        trainable_params.append((name, param.numel()))

print(f"\nTrainable parameters ({len(trainable_params)} layers):")
for name, count in trainable_params[:10]:  # Show first 10
    print(f"  {name}: {count:,} params")
if len(trainable_params) > 10:
    print(f"  ... and {len(trainable_params) - 10} more")

# Verify gradients can flow through LoRA
print("\n=== GRADIENT FLOW TEST ===")
peft_model.train()
test_input = tokenizer("Test", return_tensors="pt").to(device)
outputs = peft_model(**test_input, use_cache=False)
loss_test = outputs.logits.sum()
loss_test.backward()

grad_found = False
for name, param in peft_model.named_parameters():
    if param.requires_grad and param.grad is not None:
        grad_found = True
        print(f"✓ Gradient found in: {name} (norm: {param.grad.norm().item():.6f})")
        break

if not grad_found:
    print("✗ NO GRADIENTS FOUND - LoRA may not be properly configured!")
else:
    print("✓ Gradients are flowing correctly")

# Clean up test gradients
peft_model.zero_grad()

=== INSPECTING PREFERENCE DATASET ===

--- Example 0 ---
Prompt length: 1218 chars
Chosen length: 166 chars
Rejected length: 452 chars
Chosen == Rejected: False
Chosen first 150 chars: 1. Reasoning: Based on medical knowledge and clinical guidelines, analyzing the symptoms and presentation leads to the diagnosis.

2. Answer: A - Supr
Rejected first 150 chars: 1. Reasoning: Based on medical knowledge and clinical guidelines, analyzing the symptoms and presentation leads to the diagnosis.

2. Answer: A - Supr

--- Example 1 ---
Prompt length: 1478 chars
Chosen length: 178 chars
Rejected length: 255 chars
Chosen == Rejected: False
Chosen first 150 chars: 1. Reasoning: Based on medical knowledge and clinical guidelines, analyzing the symptoms and presentation leads to the diagnosis.

2. Answer: C - Infe
Rejected first 150 chars: :1. Reasoning: Based on medical knowledge and clinical guidelines, analyzing the symptoms and presentation leads to the diagnosis.

2. Answer: C - Inf
First differ

In [7]:
import json
from datasets import Dataset

DATASET_PATH = "/kaggle/input/preference-dataset/preference_pairs.json"

# Load the JSON file (Handling JSON Lines format)
print(f"Loading dataset from {DATASET_PATH}...")
data = []
try:
    with open(DATASET_PATH, 'r') as f:
        for line in f:
            # Parse each line as a separate JSON object
            data.append(json.loads(line))
except json.JSONDecodeError as e:
    # Fallback/check in case it was a single JSON object after all
    print(f"Failed to load as JSON Lines, trying single object: {e}")
    with open(DATASET_PATH, 'r') as f:
        f.seek(0) # Reset file pointer
        data = json.load(f)
    if not isinstance(data, list):
        data = [data] # Ensure it's a list if it was a single object

# Convert to Hugging Face Dataset
if isinstance(data, list):
    preference_dataset = Dataset.from_list(data)
elif isinstance(data, dict):
    # If it was a single dictionary for the whole dataset
    preference_dataset = Dataset.from_dict(data)
else:
    raise ValueError(f"Unexpected final data format: {type(data)}")

print(f"✓ Loaded {len(preference_dataset)} examples")
print(f"Dataset columns: {preference_dataset.column_names}\n")

print("=== DETAILED TOKEN-LEVEL ANALYSIS ===\n")

# Take first example
ex = preference_dataset[0]
prompt = ex["prompt"]
chosen = ex["chosen"]
rejected = ex["rejected"]

print(f"Prompt length: {len(prompt)} chars")
print(f"Chosen length: {len(chosen)} chars")
print(f"Rejected length: {len(rejected)} chars")
print(f"\nChosen text: {chosen}")
print(f"\nRejected text: {rejected}")

# Tokenize to see actual tokens being compared
chosen_text = prompt + chosen
rejected_text = prompt + rejected

chosen_tokens = tokenizer(chosen_text, return_tensors="pt", truncation=True, max_length=1024)
rejected_tokens = tokenizer(rejected_text, return_tensors="pt", truncation=True, max_length=1024)
prompt_tokens = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)

print(f"\nTotal tokens - Chosen: {chosen_tokens['input_ids'].shape[-1]}")
print(f"Total tokens - Rejected: {rejected_tokens['input_ids'].shape[-1]}")
print(f"Prompt tokens: {prompt_tokens['input_ids'].shape[-1]}")

# Find where they differ
chosen_ids = chosen_tokens['input_ids'][0]
rejected_ids = rejected_tokens['input_ids'][0]
prompt_len = prompt_tokens['input_ids'].shape[-1]

print(f"\nTokens being compared (after prompt):")
print(f"Chosen completion tokens: {chosen_ids.shape[0] - prompt_len}")
print(f"Rejected completion tokens: {rejected_ids.shape[0] - prompt_len}")

# Show the actual differing tokens
min_len = min(len(chosen_ids), len(rejected_ids))
diff_positions = []
for i in range(prompt_len, min_len):
    if chosen_ids[i] != rejected_ids[i]:
        diff_positions.append(i)

print(f"\nDiffering token positions: {diff_positions}")
print(f"Number of different tokens: {len(diff_positions)}")

if diff_positions:
    print("\nFirst few differences:")
    for pos in diff_positions[:10]:
        chosen_token = tokenizer.decode([chosen_ids[pos]])
        rejected_token = tokenizer.decode([rejected_ids[pos]])
        print(f"  Position {pos}: '{chosen_token}' vs '{rejected_token}'")
else:
    print("\n⚠️  WARNING: No token differences found in the first min_len tokens!")
    print("This means truncation is cutting off the actual answer difference!")

# Check if lengths differ
if len(chosen_ids) != len(rejected_ids):
    print(f"\nSequence length difference:")
    print(f"  Chosen: {len(chosen_ids)} tokens")
    print(f"  Rejected: {len(rejected_ids)} tokens")
    print(f"  Difference: {abs(len(chosen_ids) - len(rejected_ids))} tokens")

# Now test with actual model forward pass
print("\n=== TESTING ACTUAL MODEL PROBABILITIES ===")
device = next(peft_model.parameters()).device
chosen_tokens = {k: v.to(device) for k, v in chosen_tokens.items()}
rejected_tokens = {k: v.to(device) for k, v in rejected_tokens.items()}

peft_model.eval()
peft_model.config.use_cache = False
if hasattr(peft_model, "base_model") and hasattr(peft_model.base_model, "config"):
    peft_model.base_model.config.use_cache = False

with torch.no_grad():
    chosen_out = peft_model(**chosen_tokens, use_cache=False, return_dict=True)
    rejected_out = peft_model(**rejected_tokens, use_cache=False, return_dict=True)
    
    # Get log probs for completion tokens
    def get_completion_logp(outputs, input_ids, prompt_len):
        logits = outputs.logits[0]  # [L, V]
        logp = F.log_softmax(logits.float(), dim=-1)
        
        # Get token log probs for completion
        completion_logps = []
        for i in range(prompt_len, len(input_ids[0])-1):
            token_id = input_ids[0, i+1].item()
            token_logp = logp[i, token_id].item()
            token_text = tokenizer.decode([token_id])
            completion_logps.append((i+1, token_text, token_logp))
        
        return completion_logps
    
    chosen_logps = get_completion_logp(chosen_out, chosen_tokens['input_ids'], prompt_len)
    rejected_logps = get_completion_logp(rejected_out, rejected_tokens['input_ids'], prompt_len)
    
    print(f"\nChosen completion log probs (first 10 tokens):")
    for pos, token, logp in chosen_logps[:10]:
        print(f"  Pos {pos}: '{token}' -> {logp:.4f}")
    
    print(f"\nRejected completion log probs (first 10 tokens):")
    for pos, token, logp in rejected_logps[:10]:
        print(f"  Pos {pos}: '{token}' -> {logp:.4f}")
    
    chosen_total = sum(logp for _, _, logp in chosen_logps)
    rejected_total = sum(logp for _, _, logp in rejected_logps)
    
    print(f"\n{'='*60}")
    print(f"Total log prob - Chosen: {chosen_total:.6f}")
    print(f"Total log prob - Rejected: {rejected_total:.6f}")
    print(f"Raw difference: {chosen_total - rejected_total:.6f}")
    print(f"With beta=5.0: {5.0 * (chosen_total - rejected_total):.6f}")
    
    # Calculate actual loss
    diff_tensor = torch.tensor(5.0 * (chosen_total - rejected_total))
    loss = -F.logsigmoid(diff_tensor).item()
    print(f"Actual loss: {loss:.6f}")

print("\n" + "="*60)
print("DIAGNOSIS:")
if len(diff_positions) == 0 and len(chosen_ids) == len(rejected_ids):
    print("❌ CRITICAL: The sequences are IDENTICAL after tokenization!")
    print("   Truncation at 1024 tokens is cutting off the answer difference.")
    print("   SOLUTION: Increase max_len to 2048+ or use shorter prompts.")
elif abs(chosen_total - rejected_total) < 0.001:
    print("❌ CRITICAL: Model assigns nearly IDENTICAL probabilities!")
    print(f"   Difference is only {abs(chosen_total - rejected_total):.6f}")
    print("   The preference signal is too weak for gradient-based learning.")
    print("\n   RECOMMENDED SOLUTIONS:")
    print("   1. ✅ Use supervised fine-tuning (SFT) instead of GRPO/DPO")
    print("   2. Create preference pairs with more substantial differences")
    print("   3. Use a different base model that's less well-aligned")
else:
    print("✓ Model DOES see a measurable preference difference!")
    print(f"  The difference of {abs(chosen_total - rejected_total):.6f} should produce gradients.")
    print("  Zero gradients likely due to:")
    print("  1. Differences occur in tokens not covered by LoRA layers")
    print("  2. LoRA rank (r=64) may be too small")
    print("  3. Need to target different modules (add q_proj, k_proj, v_proj)")

Loading dataset from /kaggle/input/preference-dataset/preference_pairs.json...
✓ Loaded 50 examples
Dataset columns: ['prompt', 'chosen', 'rejected']

=== DETAILED TOKEN-LEVEL ANALYSIS ===

Prompt length: 1218 chars
Chosen length: 166 chars
Rejected length: 452 chars

Chosen text: 1. Reasoning: Based on medical knowledge and clinical guidelines, analyzing the symptoms and presentation leads to the diagnosis.

2. Answer: A - Supraspinatus tendon

Rejected text: 1. Reasoning: Based on medical knowledge and clinical guidelines, analyzing the symptoms and presentation leads to the diagnosis.

2. Answer: A - Supraspinatus tendon

3. Explanation: Based on medical knowledge and clinical guidelines, analyzing the symptoms and presentation leads to the diagnosis.

4. Confidence: Based on medical knowledge and clinical guidelines, analyzing the symptoms and presentation leads to the diagnosis with high confidence.


NameError: name 'tokenizer' is not defined

In [ ]:
import inspect
import trl
print("trl version:", trl.__version__)
print("SFTTrainer signature:")
print(inspect.signature(trl.SFTTrainer.__init__))
# And also show the docstring for quick hints:
print("\nSFTTrainer docstring:\n", trl.SFTTrainer.__init__.__doc__)


In [9]:
import os

print("=== CHECKING FOR SFT MODEL ===\n")

# Check the path you mentioned
sft_path = "./medical-phi3-sft-final"

if os.path.exists(sft_path):
    print(f"✓ FOUND: {sft_path}")
    
    # Check what's inside
    files = os.listdir(sft_path)
    print(f"\nContents ({len(files)} files):")
    for f in sorted(files):
        full_path = os.path.join(sft_path, f)
        if os.path.isfile(full_path):
            size_mb = os.path.getsize(full_path) / (1024*1024)
            print(f"  📄 {f} ({size_mb:.2f} MB)")
        else:
            print(f"  📁 {f}/")
    
    # Check for required files
    required_files = ['adapter_config.json', 'adapter_model.safetensors', 'tokenizer_config.json']
    missing = [f for f in required_files if f not in files]
    
    if missing:
        print(f"\n⚠️  Missing files: {missing}")
    else:
        print("\n✓ All required adapter files present!")
        print("\nYou can load this model with:")
        print(f"  peft_model = PeftModel.from_pretrained(base_model, '{sft_path}')")
    
else:
    print(f"✗ NOT FOUND: {sft_path}")
    
    # Check current directory
    print("\n=== CURRENT DIRECTORY CONTENTS ===")
    cwd = os.getcwd()
    print(f"Current directory: {cwd}")
    
    items = os.listdir('.')
    print(f"\nItems in current directory ({len(items)} total):")
    for item in sorted(items)[:30]:
        if os.path.isdir(item):
            print(f"  📁 {item}/")
        else:
            print(f"  📄 {item}")
    
    # Search for any phi3 or sft related directories
    print("\n=== SEARCHING FOR MODEL DIRECTORIES ===")
    for item in items:
        if 'phi3' in item.lower() or 'sft' in item.lower() or 'medical' in item.lower():
            print(f"  Found: {item}")

=== CHECKING FOR SFT MODEL ===

✗ NOT FOUND: ./medical-phi3-sft-final

=== CURRENT DIRECTORY CONTENTS ===
Current directory: /kaggle/working

Items in current directory (1 total):
  📁 .virtual_documents/

=== SEARCHING FOR MODEL DIRECTORIES ===


In [ ]:
!nvidia-smi -L
!nvidia-smi

import torch, gc
torch.cuda.empty_cache()
gc.collect()
print("cuda available:", torch.cuda.is_available())
print("num gpus:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i), round(torch.cuda.get_device_properties(i).total_memory/1024**3, 2), "GB")


In [ ]:
    !pip install --upgrade trl transformers